# Periodic committor experiment

## Read this before running anything

This notebook is a **launcher**, not the place where the PINN mathematics is implemented. The implementation lives in the repository so that the experiment is reproducible and not a large, duplicated notebook.

| Repository location | What it contains |
|---|---|
| `src/boundary_pinn/problems/committor/periodic_two_well.py` | The periodic two-well potential, the sets \(A,B\), the committor PDE, and the exact boundary factors. |
| `src/boundary_pinn/models.py` | The periodic neural network and the three ansatz types. |
| `src/boundary_pinn/problems/committor/training.py` | The optimisation loop and restartable checkpoints. |
| `src/boundary_pinn/problems/committor/reference.py` | The finite-difference reference solver. |
| `src/boundary_pinn/experiments/committor/periodic_two_well/comparison.py` | Runs the three methods, compares them to the reference, and writes figures. |
| `experiments/committor/periodic_two_well/configs/*.json` | The numerical choices for each run. |

The three methods are:

1. **Boundary-adapted:** the exact lift plus the first-order-normalized two-boundary factor from the paper.
2. **Naïve product:** the same exact boundary values but an unnormalized product factor; this is the ablation.
3. **Penalty:** an ordinary periodic neural network with sampled Dirichlet penalty terms.

## What the code cells do

| Cell | Purpose | Hardware | Scientific status |
|---|---|---|---|
| 1. Setup and tests | Downloads the source-controlled code, installs it, and tests periodic geometry, exact boundary values, and the boundary sampler. | CPU | Checks components only; does not train a model. |
| 2. CPU preflight | Runs the *entire* comparison pipeline with two epochs and 32/64-grid references. | CPU | Checks that training, reference solve, checkpoints, summaries, and figures work together. Not a result. |
| 3. GPU pilot | Runs all three methods for 6,000 epochs each with 128/256-grid references. | GPU | The first numerical experiment worth inspecting. Not automatically a paper figure. |

The GPU pilot does not start automatically. Run it only after checking the CPU-preflight output and deciding to use GPU time.

## What the GitHub token is for

Because the repository is private, the setup cell uses your saved `GITHUB_TOKEN` Colab Secret to fetch the latest code from the `main` branch on every setup run. It refreshes the temporary clone under `/content/boundary-adapted-pinns`; it does not change your GitHub files. The token is only for GitHub read access. It is not used for the experiment, does not consume GPU time, and is not stored in Drive or in the repository. You do not need to create another token. If the repository becomes public, the setup cell can fetch it without a token.

## 1. Setup and mathematical component tests — CPU only

Run this cell first. It creates a temporary copy of the repository in Colab's `/content` directory, installs the Python package, and runs the fast tests. No training happens here and no GPU computation is used. The printed code revision is recorded later with your run outputs.

In [ ]:
# Setup only: fetch current main, install the package, and run tests. This cell does not train a PINN.
from pathlib import Path
import base64
import subprocess

from google.colab import drive, userdata
drive.mount('/content/drive')

REPO = Path('/content/boundary-adapted-pinns')
REPO_URL = 'https://github.com/n2pakbong/boundary-adapted-pinns.git'
BRANCH = 'main'

if REPO.exists() and not (REPO / '.git').is_dir():
    raise RuntimeError(f'{REPO} exists but is not a Git clone. Rename that temporary folder, then rerun setup.')

token = None
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    # Public repositories can be cloned without a Colab Secret.
    pass

git_auth = ['git']
if token:
    auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    git_auth += ['-c', f'http.extraheader=AUTHORIZATION: basic {auth}']

try:
    if not REPO.exists():
        subprocess.run([
            *git_auth, 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO)
        ], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO), 'remote', 'set-url', 'origin', REPO_URL], check=True)
        subprocess.run([
            *git_auth, '-C', str(REPO), 'fetch', '--depth', '1', 'origin', BRANCH
        ], check=True)
        subprocess.run(['git', '-C', str(REPO), 'checkout', '-B', BRANCH, 'FETCH_HEAD'], check=True)
except subprocess.CalledProcessError as error:
    raise RuntimeError(
        'Could not fetch main from GitHub. If this repository is private, add the GITHUB_TOKEN '
        'Colab Secret and enable Notebook access. The token is only used to read the repository.'
    ) from error
finally:
    del token
    if 'auth' in globals():
        del auth

%cd /content/boundary-adapted-pinns
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.[dev]'], check=True)
subprocess.run(['python', '-m', 'pytest', '-q'], check=True)
revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Setup and mathematical component tests passed. Code revision:', revision)


## 2. CPU preflight — complete pipeline, negligible scientific cost

Run this before any GPU work. It executes the same high-level comparison command as the pilot, but with an 8-wide, 2-layer network, two epochs, 32 interior points, and 32/64-grid finite-difference references. It produces a separate timestamped Drive folder containing checkpoints, plots, and `summary.json`. Its purpose is to catch wiring failures—not to estimate a committor accurately.

In [ ]:
# CPU only: full pipeline check using the tiny preflight configuration.
from datetime import datetime
from pathlib import Path
import subprocess

preflight_name = 'preflight_' + datetime.now().strftime('%Y%m%d_%H%M%S')
PREFLIGHT_ROOT = f'/content/drive/MyDrive/boundary-adapted-pinns-runs/periodic_committor/{preflight_name}'
Path(PREFLIGHT_ROOT).mkdir(parents=True, exist_ok=False)
print('CPU preflight results:', PREFLIGHT_ROOT)

subprocess.run([
    'periodic-committor',
    'experiments/committor/periodic_two_well/configs/cpu_preflight.json',
    '--output-dir', PREFLIGHT_ROOT,
], check=True)
print('CPU preflight completed. Inspect the saved summary and figures before considering the GPU pilot.')

## 3. GPU pilot — first substantive numerical experiment

Run this only after the CPU preflight completes. This is the first experiment that uses your T4: 64-wide, 4-hidden-layer networks; 1,024 interior points per optimisation step; 6,000 epochs for each of the three methods; and 128/256-grid references. It saves every 100 epochs, so an interrupted run can be resumed from the `last.pt` checkpoints in Drive.

In [ ]:
# GPU required: full three-method committor pilot.
from datetime import datetime
from pathlib import Path
import subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError('The GPU pilot requires a T4 GPU. Select it under Runtime → Change runtime type, then rerun setup and preflight.')

run_name = 'periodic_committor_' + datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_ROOT = f'/content/drive/MyDrive/boundary-adapted-pinns-runs/periodic_committor/{run_name}'
Path(RUN_ROOT).mkdir(parents=True, exist_ok=False)
print('GPU pilot results:', RUN_ROOT)

subprocess.run([
    'periodic-committor',
    'experiments/committor/periodic_two_well/configs/gpu_pilot.json',
    '--output-dir', RUN_ROOT,
], check=True)

## 4. Inspect the finished GPU-pilot outputs

Run this only after the GPU pilot cell has completed. It displays the saved numerical summary, loss curves, three predicted committors, finite-difference references, and pointwise errors.

In [ ]:
import json
from IPython.display import Image, display

with open(f'{RUN_ROOT}/summary.json') as handle:
    print(json.dumps(json.load(handle), indent=2))
for filename in ['losses.png', 'committor_boundary_adapted.png', 'committor_naive_product.png', 'committor_penalty.png']:
    display(Image(filename=f'{RUN_ROOT}/{filename}'))

## Resume an interrupted GPU pilot

Use this only if the GPU pilot started and then stopped before completion. In a fresh Colab runtime, run the setup and CPU-preflight cells first. Then click the code cell below. It asks for the Drive folder path that the pilot printed before it began. Paste that path and press Enter. The saved checkpoints let it continue the existing run; it does not create a new run folder. Do not use this while a pilot cell is still running.

In [ ]:
# Run this only if the GPU pilot stopped before it finished.
# Paste the exact Drive folder path printed at the start of that pilot.
from pathlib import Path
import subprocess

RUN_ROOT = input('Paste the interrupted pilot Drive folder path, then press Enter: ').strip()
if not Path(RUN_ROOT, 'comparison_config.json').is_file():
    raise FileNotFoundError('That folder is missing comparison_config.json. Paste the exact folder path printed by the interrupted pilot.')

subprocess.run([
    'periodic-committor',
    'experiments/committor/periodic_two_well/configs/gpu_pilot.json',
    '--output-dir', RUN_ROOT,
    '--resume',
], check=True)
print('The interrupted pilot has resumed and completed.')